In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
PROJECT_DIR = "/content/drive/MyDrive/NutriChat-RAG/NutriChat"

%cd "$PROJECT_DIR"

/content/drive/MyDrive/NutriChat-RAG/NutriChat


In [3]:
!ls

artifacts  notebooks  nutrichat.egg-info  requirements.txt
data	   nutrichat  pyproject.toml	  results


In [4]:
!pip install -r requirements.txt
!pip install -e .

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 61.7 MB/s eta 0:00:00
Obtaining file:///content/drive/MyDrive/NutriChat-RAG/NutriChat
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for nutrichat (pyproject.toml) ... done
  Created wheel for nutrichat: filename=nutrichat-0.1.0-0.editable-py3-none-any.whl size=2811 sha256=76afdd3240aa22df0a2d7097a318af73bd2048a57a7b746f2e1a3e8b7680486c
  Stored in directory: /tmp/pip-ephem-wheel-cache-wjep1m2s/wheels/13/1f/4c/1f42e06f0c3ace164fb08724e739b140b59d7d29b5214e49d4
Successfully built nutrichat


In [5]:
import torch

from nutrichat.config import ACTIVE_CHUNKING_STRATEGY, EMBEDDING_MODEL, MIN_TOKEN_LENGTH, PDF_PAGE_OFFSET
from nutrichat.data import open_and_read_pdf, save_pickle
from nutrichat.chunking import add_chunk_ids, add_sentences_to_pages, build_chunks, filter_chunks_by_min_tokens
from nutrichat.embeddings import attach_embeddings_to_chunks, embed_chunks, load_embedding_model, tensor_from_chunk_embeddings


In [6]:
from pathlib import Path
import json
import hashlib
import numpy as np

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()
PDF_PATH = "data/nutrition_textbook.pdf"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

pages = open_and_read_pdf(PDF_PATH, page_offset=PDF_PAGE_OFFSET)
pages = add_sentences_to_pages(pages)
chunks = build_chunks(pages, strategy_name=ACTIVE_CHUNKING_STRATEGY)
chunks = filter_chunks_by_min_tokens(chunks, min_token_length=MIN_TOKEN_LENGTH)
chunks = add_chunk_ids(chunks)

embedding_model = load_embedding_model(EMBEDDING_MODEL, device=DEVICE)

index_dir = Path(f"artifacts/index_{ACTIVE_CHUNKING_STRATEGY}")
index_dir.mkdir(parents=True, exist_ok=True)

embeddings_np = embed_chunks(chunks, embedding_model, batch_size=64).astype(np.float32)
chunks_without_embeddings = []
for item in chunks:
    row = dict(item)
    row.pop("embedding", None)
    chunks_without_embeddings.append(row)

chunks_path = index_dir / "chunks.jsonl"
embeddings_path = index_dir / "embeddings.npy"
manifest_path = index_dir / "manifest.json"

with open(chunks_path, "w", encoding="utf-8") as f:
    for row in chunks_without_embeddings:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

np.save(embeddings_path, embeddings_np, allow_pickle=False)

manifest = {
    "artifact_version": 1,
    "chunking_strategy": ACTIVE_CHUNKING_STRATEGY,
    "embedding_model": EMBEDDING_MODEL,
    "normalized_embeddings": True,
    "num_chunks": len(chunks_without_embeddings),
    "embedding_shape": list(embeddings_np.shape),
    "embedding_dtype": str(embeddings_np.dtype),
    "files": {
        "chunks.jsonl": sha256_file(chunks_path),
        "embeddings.npy": sha256_file(embeddings_path),
    },
}

with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Saved index:", index_dir)

Reading PDF:   0%|          | 0/894 [00:00<?, ?it/s]

Sentence splitting:   0%|          | 0/894 [00:00<?, ?it/s]

sentence_15_no_overlap:   0%|          | 0/894 [00:00<?, ?it/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/19 [00:00<?, ?it/s]

Saved index: artifacts/index_sentence_15_no_overlap
